# Increment 10 - Static Mechanical Properties and Rock-Strength Scenarios
**Poseidon 1D MEM | Mikael Elgo | p2mem 0.10.0**

Tier C: screening-level, uncalibrated educational work.

This notebook reproduces a **hypothetical sandstone-analogue experiment** from the packaged Increment 9 dynamic results. It does not identify sandstone in Poseidon, calibrate static properties, or produce field-approved strength values. The field branch remains withheld.

**Start:** put `Poseidon_1D_MEM_Increment_10.zip` in `My Drive/Poseidon_1D_MEM/`, open this notebook in Colab, then **Runtime > Run all**. No raw LAS upload is required for this increment. Do not run the historical notebooks over the new package.

## 1. Choose the release and mode
`reproduce` recalculates Increment 10 from the packaged derived data and validates it. `review` validates the supplied results. Both run all regression tests in a disposable copy and execute independent numerical checks.

If your ZIP was renamed by a download, set its exact path in `ZIP_PATH`. `PROJECT_ROOT` is for an already extracted, verified local checkout. Leave it empty in Colab.

In [ ]:
MODE = "reproduce"  # "reproduce" or "review"
ZIP_PATH = ""       # Example: /content/drive/MyDrive/Poseidon_1D_MEM/Poseidon_1D_MEM_Increment_10.zip
PROJECT_ROOT = ""   # Local use only, or an explicit existing verified extraction
INSTALL_DEPENDENCIES = True


## 2. Extract a fresh verified working copy
The original ZIP stays unchanged. Extraction rejects unsafe paths, duplicate entries, symlinks and files absent from the release ledger. Every fresh extraction gets its own folder, preventing stale-tree collisions.

In [ ]:
"""Standard-library-only safe extraction and file-ledger checks for Colab."""
import hashlib
from pathlib import Path, PurePosixPath
import shutil
import stat
import tempfile
from zipfile import ZipFile

LEDGER_NAME='INCREMENT_10_SHA256SUMS.txt'


def verify_tree(root):
    root=Path(root)
    if root.is_symlink() or not root.is_dir():raise ValueError('Project root must be a regular directory')
    seen=set();ledger=root/LEDGER_NAME
    if not ledger.is_file() or ledger.is_symlink():raise ValueError('Increment 10 ledger missing')
    for line in ledger.read_text(encoding='utf-8').splitlines():
        if not line or line.startswith('#'):continue
        try:digest,name=line.split('  ',1)
        except ValueError as exc:raise ValueError('Malformed ledger') from exc
        p=PurePosixPath(name)
        if (p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name
            or name in seen or not name or name==LEDGER_NAME):raise ValueError('Unsafe ledger path')
        if len(digest)!=64 or any(c not in '0123456789abcdef' for c in digest):raise ValueError('Malformed checksum')
        seen.add(name);target=root/name
        if any(root.joinpath(*p.parts[:i]).is_symlink() for i in range(1,len(p.parts)+1)):
            raise ValueError('Symlink in release path')
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest()!=digest:
            raise ValueError('Missing or changed release file: '+name)
    if not seen:raise ValueError('Empty ledger')
    return seen


def extract_release(archive,parent=None):
    archive=Path(archive)
    if not archive.is_file():raise FileNotFoundError('Set ZIP_PATH to the exact Increment 10 ZIP')
    parent=Path(parent) if parent is not None else Path(tempfile.gettempdir())
    parent.mkdir(parents=True,exist_ok=True);staging=None
    try:
        with ZipFile(archive) as z:
            names=set();total=0
            for info in z.infolist():
                name=info.filename;p=PurePosixPath(name);total+=info.file_size
                if (p.is_absolute() or '..' in p.parts or '\\' in name or ':' in name or p.as_posix()!=name
                    or name in names or not name or info.is_dir() or stat.S_ISLNK(info.external_attr>>16)):
                    raise ValueError('Unsafe or duplicate ZIP entry: '+name)
                names.add(name)
            if total>300_000_000 or len(names)>10000:raise ValueError('Unexpected package size')
            staging=Path(tempfile.mkdtemp(prefix='poseidon_inc10_',dir=parent))
            z.extractall(staging)
        listed=verify_tree(staging)
        if names!=listed|{LEDGER_NAME}:raise ValueError('ZIP and ledger inventories differ')
        return staging
    except BaseException:
        if staging is not None:shutil.rmtree(staging,ignore_errors=True)
        raise


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if MODE not in ("review", "reproduce"):
    raise ValueError("MODE must be review or reproduce")
if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT).expanduser().resolve()
    verify_tree(project_root)
else:
    if ZIP_PATH:
        archive = Path(ZIP_PATH).expanduser()
    else:
        candidates = [Path("/content/drive/MyDrive/Poseidon_1D_MEM/Poseidon_1D_MEM_Increment_10.zip"),
                      Path("/content/drive/MyDrive/Poseidon_1D_MEM_Increment_10.zip")]
        available = [p for p in candidates if p.is_file()]
        if len(available) != 1:
            raise FileNotFoundError("Set ZIP_PATH to the exact Increment 10 ZIP; no unique default was found.")
        archive = available[0]
    print("Archive SHA-256:", hashlib.sha256(archive.read_bytes()).hexdigest())
    project_root = extract_release(archive, "/content" if IN_COLAB else None)
print("Verified working folder:", project_root)


## 3. Install the declared dependencies
The scientific workflow needs NumPy, PyYAML and Matplotlib; pytest verifies the release. The notebook runs the package through a fresh Python subprocess, so an old `p2mem` import in your kernel cannot silently select a previous increment.

In [ ]:
import subprocess
import sys
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-q", "-e", ".[dev]"],
                   cwd=project_root, check=True)
else:
    print("Using the already installed declared dependencies.")


## 4. Scientific method and eligibility
The static analogue uses Mahdi & Alrazzaq (2024), Eq. 8:

$$E_{static}\,[GPa] = 0.3655\,(E_{dynamic}\,[GPa])^{1.0959}.$$

The reported predictor span is **17.90-43.45 GPa**. This notebook withholds outputs outside it. Being within that span establishes numerical support only; it does not establish a material or calibration match.

For the strength experiment, Chang et al. (2006), Table 1 Eq. 8 gives:

$$UCS\,[MPa] = 46.2\,\exp(0.027 E_{static}\,[GPa]).$$

The source table does not identify an originating region/reference for that particular equation. We therefore retain it as an explicitly hypothetical response law, never a field-eligible UCS correlation.

Sources and limitations: `config/mechanics_correlations.json` and `INCREMENT_10_SCIENTIFIC_REPORT.md`.

## 5. Assumed parameters and derived mechanics
The reference experiment uses **static nu = 0.25**, **phi = 30 degrees**, and **T0/UCS = 0.05**. These are project-selected experiment values, not measured properties or literature uncertainty bounds. Dynamic nu is not silently copied to static nu.

$$G=\frac{E}{2(1+\nu)},\quad K=\frac{E}{3(1-2\nu)},\quad c=\frac{UCS(1-\sin\phi)}{2\cos\phi},\quad \mu=\tan\phi.$$

Seven cases vary nu, phi or the tensile ratio one at a time. The four full profiles hold the reference experiment at every original sample; the scenario summary tables and config define the other cases exactly. No P10/P50/P90 or best-estimate claim is made.

## 6. Reproduce, independently verify and run all tests
This is the longest cell. It prints the test result after the disposable test run completes. An error stops execution; it is never treated as a passed gate. Field eligibility remaining withheld is the intended scientific result, not a test failure.

In [ ]:
command = [sys.executable, str(project_root / "scripts/run_increment_10.py"), "--mode", MODE]
completed = subprocess.run(command, cwd=project_root, text=True, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, check=False)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError("Increment 10 did not pass. See the output above and run_records/increment_10_pytest.log.")
verify_tree(project_root)


## 7. Read coverage before reading the curves
Three outcomes are distinct: dynamic input missing, outside the analogue's predictor span, and numerically supported conditional analogue. **None is field-eligible mechanics in this release.** The denominator below is the whole original LAS population, not formation thickness.

In [ ]:
import csv
import json
from IPython.display import display, Markdown, Image
report = json.loads((project_root / "run_records/increment_10_run.json").read_text())
output_root = project_root / "outputs" / report["output_directory"]
manifest = json.loads((output_root / "mechanics_manifest.json").read_text())
lines = ["| Well | Original samples | Valid dynamic E | Analogue numeric support | Field status |",
         "|---|---:|---:|---:|---|"]
for row in manifest["coverage"]:
    lines.append(f"| {row['well_key']} | {row['n_original']:,} | {row['n_upstream_dynamic']:,} | {row['n_analogue_numeric']:,} | WITHHELD |")
display(Markdown("\n".join(lines)))
display(Markdown(f"**Verification passed:** {report['tests_passed']:,} tests; "
                 f"{report['historical_tests_passed']:,} historical tests retained; "
                 f"{report['independent_checks']['checked_rows']} independently checked numeric rows."))


## 8. Inspect the reference experiment
Gray dynamic E is shown for context. Colored profiles are conditional analogue outputs. Blank intervals remain blank; zooming does not create continuous support.

In [ ]:
for well in ("Poseidon_2", "Boreas_1", "Poseidon_North_1", "Proteus_1ST2"):
    display(Image(filename=str(output_root / f"{well}_mechanics_qc.png")))


## 9. Compare the seven experiment cases
These medians use each property's supported original samples. Parameter-case spread is a numerical experiment, not a confidence interval. Because E and UCS do not depend on nu, phi or T0 in the selected chain, their values should not change across these seven cases.

In [ ]:
with (output_root / "mechanics_summary.csv").open(newline="") as stream:
    rows = list(csv.DictReader(stream))
selected = [r for r in rows if r["well_key"] == "Poseidon_2" and r["property"] in
            ("G_static_scenario_GPa", "K_static_scenario_GPa", "cohesion_scenario_MPa", "T0_assumed_MPa")]
lines = ["| Case | Property | Median | Change from reference |", "|---|---|---:|---:|"]
for r in selected:
    median = f"{float(r['median']):.4f}" if r['median'] else "unavailable"
    delta = f"{float(r['median_change_from_reference']):+.4f}" if r['median_change_from_reference'] else "unavailable"
    lines.append(f"| {r['case_id']} | {r['property']} | {median} | {delta} |")
display(Markdown("\n".join(lines)))


## 10. Formation context and handoff
`formation_interval_summary.csv` uses accepted marker-to-marker intervals only, preserves the inherited depth and well-identity status, and uses half-open MD intervals. It does not assign lithology. North 1 and Proteus receive no invented formation summaries.

Increment 11 may consume these values only as explicit conditional experiments, with compatible pressure/stress assumptions and uncertainty labels. This release contains **no horizontal stress or WBS implementation**.

## 11. Save your results
The final ZIP includes the output bundle, config/source registry, scientific report and run records. In Colab it is saved to `My Drive/Poseidon_1D_MEM/`. It is a results archive, not a replacement for the cumulative source release.

In [ ]:
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
save_parent = Path("/content/drive/MyDrive/Poseidon_1D_MEM") if IN_COLAB else project_root.parent / "increment10_results"
save_parent.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
result_zip = save_parent / f"Poseidon_Increment_10_results_{stamp}.zip"
with ZipFile(result_zip, "x", ZIP_DEFLATED) as bundle:
    for path in sorted(output_root.iterdir()):
        if path.is_file(): bundle.write(path, "outputs/10_static_mechanics_strength/" + path.name)
    for name in ("config/mechanics_scenarios.json", "config/mechanics_correlations.json", "INCREMENT_10_SCIENTIFIC_REPORT.md"):
        bundle.write(project_root / name, name)
    for path in sorted((project_root / "run_records").glob("increment_10*")):
        if path.is_file(): bundle.write(path, "run_records/" + path.name)
print("Saved:", result_zip)
print("COMPLETE: verified conditional mechanics. Field mechanics remain withheld. Increment 11 has not started.")
